# GLM Pricing: Frequency × Severity = Pure Premium

**Dataset**: French Motor Third-Party Liability (freMTPL2), ~678k policies  
**Methods**: Poisson GLM (frequency), Gamma GLM (severity), pure premium segmentation  
**Key question**: What is the expected cost to insure each risk for one policy-year?


## 1. The Frequency-Severity Framework

The expected total claims cost for policy $i$ over one year is:

$$E[\text{Total Cost}_i] = \underbrace{E[N_i]}_\text{frequency} \times \underbrace{E[S_i \mid N_i > 0]}_\text{severity}$$

We model each component separately:

### Frequency Model (Poisson GLM)

Claim count $N_i \sim \text{Poisson}(\mu_i)$ where:

$$\log\left(\frac{\mu_i}{e_i}\right) = \mathbf{x}_i^\top \boldsymbol{\beta}$$

$e_i$ = exposure (policy-years). This gives:

$$\mu_i = e_i \cdot \exp(\mathbf{x}_i^\top \boldsymbol{\beta})$$

The **log link** ensures $\mu_i > 0$. The **exposure offset** adjusts for policies active less than a full year.

### Severity Model (Gamma GLM)

Claim severity $S_i \mid N_i > 0 \sim \text{Gamma}(\alpha, \theta_i)$ where:

$$\log(E[S_i]) = \mathbf{x}_i^\top \boldsymbol{\gamma}$$

The Gamma distribution is appropriate because:
- Claim costs are strictly positive
- Variance $\propto$ mean$^2$ (constant coefficient of variation)
- Right-skewed, matching empirical claim distributions

### Pure Premium

$$\text{Pure Premium}_i = \hat{\mu}_i / e_i \times \hat{S}_i = \hat{f}_i \times \hat{s}_i$$

This is the **risk-based rate**: the expected annual cost before loading for expenses and profit.


## 2. Exploratory Data Analysis

In [ ]:
import sys
sys.path.insert(0, 'actuarial')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from frequency import load_fremtpl, FrequencyModel, eda_summary
from severity import SeverityModel, load_severity_data, pure_premium_analysis

freq_path = 'data/fremtpl/freMTPL2freq.csv'
sev_path  = 'data/fremtpl/freMTPL2sev.csv'

df = load_fremtpl(freq_path)
eda_summary(df)

### Key EDA Findings

- **Young drivers (18-25)**: frequency 1.74x the portfolio average — classic high-risk segment
- **New vehicles (0-1yr)**: frequency 1.63x — new vehicles are driven more aggressively and are involved in more accidents
- **Area F (most urban)**: frequency 1.38x — urban congestion increases accident exposure
- **BonusMalus**: will be the dominant predictor — it summarizes the policyholder's claims history


## 3. Frequency Model (Poisson GLM)

In [ ]:
print('Fitting Poisson GLM...')
freq_model = FrequencyModel()
freq_model.fit(df)
freq_model.summary()

In [ ]:
metrics = freq_model.evaluate()
fig = freq_model.plot()
plt.show()

### Interpreting the Coefficients

**LogBonusMalus** (relativity ≈ 5.4): the single most important predictor. A policyholder with BonusMalus score at the 95th percentile has 5× the expected frequency of one at the median. This validates the French bonus-malus system as an effective risk classifier.

**Driver age**: the base category is 18-25 (highest risk). Older drivers show lower frequency:
- 26-35: 0.83× (17% lower)
- 66+: 1.28× — the U-shaped age curve appears; senior drivers have slightly higher frequency

**Area**: not statistically significant after controlling for `LogDensity`. Population density is a better continuous proxy for geographic risk than administrative area codes.

**Gini = 24.5%**: reasonable discrimination for a frequency GLM. Pure-premium Gini on full datasets typically ranges 15-35%.


## 4. Severity Model (Gamma GLM)

In [ ]:
print('Loading severity data...')
sev_df = load_severity_data(freq_path, sev_path)

print('\nFitting Gamma GLM...')
sev_model = SeverityModel()
sev_model.fit(sev_df)
sev_model.summary()

In [ ]:
sev_metrics = sev_model.evaluate()
fig = sev_model.plot()
plt.show()

### Interpreting Severity Results

**Key finding**: severity drivers are mostly different from frequency drivers.

- **Driver age** strongly affects severity: young drivers (18-25) have dramatically higher severity (base category). The relativity for 51-65 is 0.44 — middle-aged drivers have lower-severity claims, likely because they avoid high-speed collisions.

- **BonusMalus is not significant for severity** (p=0.31). High-BM policyholders have more claims, but not more expensive ones. This is an important insight: the bonus-malus system captures frequency risk but not severity risk.

- **Vehicle characteristics** (power, age, fuel type) are not significant for severity. This suggests severity is driven more by how people drive than what they drive.

This frequency-severity divergence is exactly why separating the two models is important — a single combined model would miss these distinct risk dynamics.


## 5. Pure Premium = Frequency × Severity

In [ ]:
pp_df = pure_premium_analysis(
    freq_model, sev_model, df,
    save_path='pure_premium.png'
)

### Pure Premium Takeaways

| Age Band | Frequency | Severity | Pure Premium | vs Average |
|----------|-----------|----------|--------------|------------|
| 18-25    | HIGH      | HIGH     | $833         | 3.4x       |
| 26-35    | Average   | Average  | $226         | 0.9x       |
| 51-65    | Low       | LOW      | $186         | 0.8x       |

Young drivers are expensive for **two independent reasons** — they have both more accidents AND more expensive accidents. This justifies large age surcharges in rating plans.

New vehicles (0-1yr) at 1.5x average: high frequency from aggressive driving of new cars, partially offset by lower-than-average severity (newer cars have better safety features).
